# 01 — Bronze ingestion: master-data direct-upload landing

| Field | Value |
| ----- | ----- |
| **Sprint** | Sprint 09 — T2.3 |
| **Layer** | `bronze/master-data/` |
| **Source** | `Files/master-data/*.csv` (direct-upload lakehouse landing folder) |
| **Target** | `Tables/bronze/master-data/<table>/` (Delta, managed) |
| **Governance** | [ADR-0015](../../../docs/adr/0015-skip-sql-for-mvp-demo.md) (SQL-free MVP path), [ADR-0016](../../../docs/adr/0016-no-phi-in-mvp-demo-scope.md) (no-PHI demo scope) |
| **Design spec** | [§3.2 Master data loading pipeline](../../../docs/superpowers/specs/2026-06-29-sprint09-master-data-capacity-dashboard-design.md) and [sprint-09 §2.2 Notebook 01](../../../docs/sprints/sprint-09-master-data-simulation-and-capacity-dashboard.md) |

## Purpose

Ingest the 9 master-data CSV fixtures **as-is** into the bronze zone with a lineage
stamp. No validation, no schema coercion, no PHI screening at this layer — those
gates live in the silver notebook (`02_silver_master_data.ipynb`).

The **direct-upload** path (per ADR-0015 and sprint-09 v2 RB-12 resolution) means
CSVs land in the lakehouse `Files/master-data/` folder manually or via a lakehouse
REST call — no SQL source, no Dataflow Gen2, no orchestrated copy activity.

## Source fixture

The reference CSVs live in the repo at
`docs/reviews/2026-06-29-ama-capacity-metadata-review/*.csv` and are uploaded
verbatim to the lakehouse `Files/master-data/` folder before this notebook runs.

## Tables covered (9 loadable + 1 reference workbook)

See the `TABLES` registry in the config cell below. The tenth artefact
(`SwissHospital_MasterData.xlsx`) is the source-of-truth workbook and is **not**
loaded as a bronze Delta table — it is retained for manual gap-fill only.

In [ ]:
# Parameters (Fabric injects overrides via the papermill-compatible 'parameters' tag).
# %%capture-equivalent is expressed here as a plain code cell tagged 'parameters'.
target_lakehouse = 'lh_ihzhhpf_sit'
hospital_csv_path = 'Files/master-data'                    # lakehouse-relative source folder
bronze_root = 'Tables/bronze/master-data'                  # lakehouse-relative bronze root
run_id = 'run-manual-local'                                # overridden by pipeline / notebookutils

In [ ]:
# Master-data table registry — 9 loadable CSV tables + 1 reference workbook.
# Fixture source: docs/reviews/2026-06-29-ama-capacity-metadata-review/*.csv
# Design spec: docs/superpowers/specs/2026-06-29-sprint09-master-data-capacity-dashboard-design.md §3.2
TABLES = [
    # (csv_filename, table_name, primary_key, foreign_keys, residency_source_col, quality_source_col)
    ('01_dim_hospital.csv',                       'dim_hospital',                             ['hospital_id'],                       {},                                                                                      'residency_tag', 'beds_quality'),
    ('02_dim_specialty.csv',                      'dim_specialty',                            ['specialty_hospital_id'],             {'hospital_id': 'dim_hospital'},                                                         None,             'data_quality'),
    ('03_dim_hospital_service.csv',               'dim_hospital_service',                     ['service_id'],                        {'hospital_id': 'dim_hospital', 'specialty_id': 'dim_specialty'},                        None,             'data_quality'),
    ('04_dim_disease.csv',                        'dim_disease',                              ['disease_id'],                        {},                                                                                      None,             None),
    ('05_dim_treatment.csv',                      'dim_treatment',                            ['treatment_id'],                      {'disease_id': 'dim_disease'},                                                           None,             None),
    ('06_dim_drg.csv',                            'dim_drg',                                  ['drg_code'],                          {'disease_id': 'dim_disease'},                                                           None,             None),
    ('07_dim_ward_capacityunit.csv',              'dim_ward_capacityunit',                    ['ward_id'],                           {'hospital_id': 'dim_hospital', 'specialty_id': 'dim_specialty'},                        None,             'bed_count_quality'),
    ('08_fact_capacity_baseline.csv',             'fact_capacity_baseline',                   ['hospital_id', 'metric'],             {'hospital_id': 'dim_hospital'},                                                         None,             'data_quality'),
    ('09_map_disease_treatment_specialty_service.csv', 'map_disease_treatment_specialty_service', ['map_id'],                       {'hospital_id': 'dim_hospital', 'disease_id': 'dim_disease', 'treatment_id': 'dim_treatment', 'drg_code': 'dim_drg', 'specialty_id': 'dim_specialty'}, None, 'data_quality'),
]
# Reference-only artefact (not loaded as a Delta table):
REFERENCE_WORKBOOK = 'SwissHospital_MasterData.xlsx'

In [ ]:
from datetime import datetime, timezone
from pyspark.sql import functions as F

def _load_timestamp() -> str:
    return datetime.now(timezone.utc).strftime('%Y-%m-%dT%H:%M:%SZ')

def load_csv_to_bronze(csv_filename: str, table_name: str) -> int:
    """Read one CSV from Files/master-data and write it verbatim to bronze Delta."""
    src = f'{hospital_csv_path}/{csv_filename}'
    tgt = f'{bronze_root}/{table_name}'
    load_ts = _load_timestamp()
    lineage = f'{csv_filename}:{load_ts}'

    df = (spark.read
              .option('header', 'true')
              .option('inferSchema', 'true')
              .option('multiLine', 'true')
              .option('escape', '"')
              .csv(src))
    df = df.withColumn('_lineage_ref', F.lit(lineage))
    (df.write
         .format('delta')
         .mode('overwrite')
         .option('overwriteSchema', 'true')
         .save(tgt))
    return df.count()

In [ ]:
# Ingest all 9 tables and collect row counts.
results = []
for csv_filename, table_name, _pk, _fks, _res, _qual in TABLES:
    try:
        n = load_csv_to_bronze(csv_filename, table_name)
        results.append((table_name, n, 'ok', None))
    except Exception as exc:  # surface but do not swallow; bronze is fail-fast at file level
        results.append((table_name, 0, 'error', str(exc)))
        raise

In [ ]:
# Summary — row count per bronze table.
print('Bronze ingestion summary (run_id=%s)' % run_id)
print('-' * 72)
for table_name, n, status, err in results:
    print(f'{table_name:<48s} rows={n:<8d} status={status}')
print('-' * 72)
print(f'Reference-only workbook (not loaded): {REFERENCE_WORKBOOK}')